# Aarya Sequential Trace — Quantitative Metrics Extraction

Extracts quantitative metrics from the 3 sequential fault buckets of the aarya run.

**Changes active in this run:**
- Full span input + output passed (no truncation)
- Batch size: 6 spans per LLM call
- Detection: indicative behaviour counts (loosened from explicit-confirmation-only)
- `detection_success` derived from resolved LLM detection time, not just `bucket.detected_at`

## 1. Imports & Path Setup

In [1]:
import sys
import os
import json
import asyncio
import logging
from pathlib import Path
from pprint import pprint

# Resolve certifier root
CERTIFIER_ROOT = Path.cwd()
while CERTIFIER_ROOT.name != "certifier" and CERTIFIER_ROOT != CERTIFIER_ROOT.parent:
    CERTIFIER_ROOT = CERTIFIER_ROOT.parent
if str(CERTIFIER_ROOT) not in sys.path:
    sys.path.insert(0, str(CERTIFIER_ROOT))

print(f"Certifier root: {CERTIFIER_ROOT}")

from metrics_extractor.scripts.metrics_extractor_from_trace import TraceMetricsExtractor
from metrics_extractor.schema.metrics_model import LLMQuantitativeExtraction

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.StreamHandler()],
)
print("Imports OK.")

Certifier root: c:\Users\meemankgupta\Music\Project\infosys\certifier
Imports OK.


## 2. Fault Buckets Directory & Files

In [2]:
FAULT_BUCKETS_DIR = Path(
    r"C:\Users\meemankgupta\Music\Project\infosys\certifier"
    r"\data\output\08-05-26-aarya\1960bc89"
    r"\f6152c39-sequential-defaults"
    r"\f6152c39-27c4-4601-a5e8-bee80da8ec78\fault_buckets"
)

bucket_files = sorted(
    f for f in FAULT_BUCKETS_DIR.glob("raw_trace_sequential_bucket_*.json")
)

print(f"Found {len(bucket_files)} fault bucket(s):")
for f in bucket_files:
    print(f"  {f.name}")

Found 3 fault bucket(s):
  raw_trace_sequential_bucket_pod-cpu-hog.json
  raw_trace_sequential_bucket_pod-memory-hog.json
  raw_trace_sequential_bucket_pod-network-loss.json


## 3. Extract Quantitative Metrics for Each Fault

Each bucket file is in `{events: [...], ...bucket_metadata}` format.
`TraceMetricsExtractor.load_trace_file()` automatically extracts the bucket metadata
(injection timestamps, ground truth, fault name) and the span list.

In [3]:
results = {}  # fault_name -> LLMQuantitativeExtraction

for bucket_file in bucket_files:
    fault_name = bucket_file.stem.replace("raw_trace_sequential_bucket_", "")
    print(f"\n{'='*60}")
    print(f"Processing: {fault_name}")
    print(f"{'='*60}")

    extractor = TraceMetricsExtractor()
    spans = extractor.load_trace_file(str(bucket_file))

    print(f"  Spans loaded: {len(spans)}")
    print(f"  Bucket metadata: fault_id={extractor.bucket_metadata.get('fault_id')}, "
          f"injection={extractor.bucket_metadata.get('injection_timestamp')}")
    print(f"  Batch size: {extractor.BATCH_SIZE}")
    print(f"  Batches: {len(extractor._create_batches(spans))}")

    quant = await extractor.extract_quantitative_metrics(spans)
    results[fault_name] = quant

    print(f"\n  --- Result for {fault_name} ---")
    print(f"  detection_success:         {quant.detection_success}")
    print(f"  fault_detected:            {quant.fault_detected}")
    print(f"  detected_fault_type:       {quant.detected_fault_type}")
    print(f"  fault_target_service:      {quant.fault_target_service}")
    print(f"  fault_namespace:           {quant.fault_namespace}")
    print(f"  fault_injection_time:      {quant.fault_injection_time}")
    print(f"  agent_fault_detection_time:{quant.agent_fault_detection_time}")
    print(f"  agent_fault_mitigation_time:{quant.agent_fault_mitigation_time}")
    print(f"  time_to_detect (s):        {quant.time_to_detect}")
    print(f"  time_to_mitigate (s):      {quant.time_to_mitigate}")
    print(f"  input_tokens:              {quant.input_tokens}")
    print(f"  output_tokens:             {quant.output_tokens}")
    print(f"  trajectory_steps:          {quant.trajectory_steps}")
    print(f"  tool_calls count:          {len(quant.tool_calls)}")
    print(f"  tool_selection_accuracy:   {quant.tool_selection_accuracy}")
    print(f"  pii_detection:             {quant.pii_detection}")

print("\nAll extractions complete.")

2026-05-11 20:58:03,414 - [metrics_extractor_from_trace.py : load_trace_file : 286] - INFO - Loaded bucket metadata from trace file: fault_id=pod-cpu-hog, fault_name=pod-cpu-hog



Processing: pod-cpu-hog
  Spans loaded: 13
  Bucket metadata: fault_id=pod-cpu-hog, injection=2026-05-08T05:22:22.000Z
  Batch size: 6
  Batches: 3


2026-05-11 20:58:04,482 - [azure_openai_util.py : get_clients : 101] - INFO - Created Azure OpenAI client for model: embedding_model
2026-05-11 20:58:04,810 - [azure_openai_util.py : get_clients : 101] - INFO - Created Azure OpenAI client for model: gpt-4o
2026-05-11 20:58:05,167 - [azure_openai_util.py : get_clients : 101] - INFO - Created Azure OpenAI client for model: gpt-5.2
2026-05-11 20:58:05,168 - [azure_openai_util.py : __init__ : 44] - INFO - AzureLLMClient initialized successfully
2026-05-11 20:58:05,169 - [metrics_extractor_from_trace.py : extract_quantitative_metrics : 705] - INFO - Processing 13 spans in 3 batches
2026-05-11 20:58:05,169 - [metrics_extractor_from_trace.py : extract_quantitative_metrics : 709] - INFO - Processing quantitative batch 1/3
2026-05-11 20:58:05,173 - [azure_openai_util.py : _get_or_create_agent : 135] - WARNING - No specific client found for model 'gpt-4o_structured'. Using default client.
2026-05-11 20:58:19,925 - [metrics_extractor_from_trace.p


  --- Result for pod-cpu-hog ---
  detection_success:         1
  fault_detected:            No explicit chaos fault was confirmed. The agent observed transient startup probe failures (Unhealthy events for carts-6b5d88f967-qpgk7 on /health with context deadline exceeded and connection refused) and considered the flash-agent pod as a potential external-disturbance/chaos framework but concluded the namespace is healthy or degraded based on pod status and events, without identifying an active injected fault.
  detected_fault_type:       None
  fault_target_service:      name=carts
  fault_namespace:           sock-shop
  fault_injection_time:      2026-05-08T05:22:22.000Z
  agent_fault_detection_time:2026-05-08T05:22:43.105Z
  agent_fault_mitigation_time:None
  time_to_detect (s):        21.11
  time_to_mitigate (s):      None
  input_tokens:              38603
  output_tokens:             12024
  trajectory_steps:          13
  tool_calls count:          46
  tool_selection_accuracy:   

2026-05-11 21:00:09,786 - [metrics_extractor_from_trace.py : extract_quantitative_metrics : 709] - INFO - Processing quantitative batch 2/3
2026-05-11 21:00:09,792 - [azure_openai_util.py : _get_or_create_agent : 135] - WARNING - No specific client found for model 'gpt-4o_structured'. Using default client.
2026-05-11 21:00:25,374 - [metrics_extractor_from_trace.py : extract_quantitative_metrics : 709] - INFO - Processing quantitative batch 3/3
2026-05-11 21:00:25,382 - [azure_openai_util.py : _get_or_create_agent : 135] - WARNING - No specific client found for model 'gpt-4o_structured'. Using default client.
2026-05-11 21:00:30,963 - [metrics_extractor_from_trace.py : extract_quantitative_metrics : 715] - INFO - Aggregating quantitative metrics from all batches
2026-05-11 21:00:30,964 - [metrics_extractor_from_trace.py : _aggregate_quantitative_metrics : 622] - INFO - Identifying detection and mitigation spans using LLM...
2026-05-11 21:00:41,714 - [metrics_extractor_from_trace.py : _i


  --- Result for pod-memory-hog ---
  detection_success:         1
  fault_detected:            No explicit chaos fault type was labeled by the agent in this batch. The agent observed transient startup probe failures on the carts pod (HTTP /health timeouts and connection refused), single restarts on the user and flash-agent pods, and the presence of a non-standard flash-agent pod in namespace sock-shop, classifying overall namespace health as degraded with indications of possible external disturbance or experimentation.
  detected_fault_type:       None
  fault_target_service:      name=orders
  fault_namespace:           sock-shop
  fault_injection_time:      2026-05-08T05:36:49.000Z
  agent_fault_detection_time:2026-05-08T05:38:16.733Z
  agent_fault_mitigation_time:None
  time_to_detect (s):        87.73
  time_to_mitigate (s):      None
  input_tokens:              28447
  output_tokens:             12519
  trajectory_steps:          13
  tool_calls count:          38
  tool_select

2026-05-11 21:02:07,232 - [metrics_extractor_from_trace.py : extract_quantitative_metrics : 709] - INFO - Processing quantitative batch 2/3
2026-05-11 21:02:07,239 - [azure_openai_util.py : _get_or_create_agent : 135] - WARNING - No specific client found for model 'gpt-4o_structured'. Using default client.
2026-05-11 21:02:27,708 - [metrics_extractor_from_trace.py : extract_quantitative_metrics : 709] - INFO - Processing quantitative batch 3/3
2026-05-11 21:02:27,716 - [azure_openai_util.py : _get_or_create_agent : 135] - WARNING - No specific client found for model 'gpt-4o_structured'. Using default client.
2026-05-11 21:02:50,643 - [metrics_extractor_from_trace.py : extract_quantitative_metrics : 715] - INFO - Aggregating quantitative metrics from all batches
2026-05-11 21:02:50,645 - [metrics_extractor_from_trace.py : _aggregate_quantitative_metrics : 622] - INFO - Identifying detection and mitigation spans using LLM...
2026-05-11 21:03:30,050 - [metrics_extractor_from_trace.py : _i


  --- Result for pod-network-loss ---
  detection_success:         1
  fault_detected:            User service in namespace sock-shop experiencing an application dependency failure: pod user-5bc44bf4bf-57qwz is Running but not Ready, with logs repeatedly showing 'no reachable servers' when connecting to backend MongoDB at MONGO_HOST=user-db:27017. Overall namespace health classified as degraded with health_score=82, 16 total pods, 15 healthy, 1 unhealthy. Flash-agent pod identified as an external analysis/automation agent running in sock-shop; metrics-server (pods.metrics.k8s.io) access forbidden for serviceaccount system:serviceaccount:litmus:mcp-server and Prometheus tools list_metrics/execute_query unavailable to the MCP server, creating observability gaps.
  detected_fault_type:       None
  fault_target_service:      name=user-db
  fault_namespace:           sock-shop
  fault_injection_time:      2026-05-08T05:29:37.000Z
  agent_fault_detection_time:2026-05-08T05:31:33.935Z
  age

## 4. Summary Table — All 3 Faults

In [4]:
print(f"{'Fault':<22} {'Detected':>9} {'TTD(s)':>9} {'TTM(s)':>9} "
      f"{'Tokens-In':>11} {'Tokens-Out':>11} {'Tools':>7} {'ToolAcc':>9}")
print("-" * 90)
for fault_name, q in results.items():
    print(
        f"{fault_name:<22} "
        f"{str(q.detection_success):>9} "
        f"{str(q.time_to_detect or 'N/A'):>9} "
        f"{str(q.time_to_mitigate or 'N/A'):>9} "
        f"{str(q.input_tokens or 0):>11} "
        f"{str(q.output_tokens or 0):>11} "
        f"{len(q.tool_calls):>7} "
        f"{str(q.tool_selection_accuracy or 'N/A'):>9}"
    )

Fault                   Detected    TTD(s)    TTM(s)   Tokens-In  Tokens-Out   Tools   ToolAcc
------------------------------------------------------------------------------------------
pod-cpu-hog                    1     21.11       N/A       38603       12024      46    0.6304
pod-memory-hog                 1     87.73       N/A       28447       12519      38       0.5
pod-network-loss               1    116.94       N/A       72760       13816      62    0.6667


## 5. Full JSON Output Per Fault

In [5]:
for fault_name, q in results.items():
    print(f"\n{'='*60}")
    print(f"FULL JSON — {fault_name}")
    print(f"{'='*60}")
    print(q.model_dump_json(indent=2))


FULL JSON — pod-cpu-hog
{
  "agent_name": "demoinfra2",
  "agent_id": "1960bc89-361a-4fcb-af86-699113f09ec9",
  "agent_version": "3.0.0",
  "experiment_id": "f6152c39-27c4-4601-a5e8-bee80da8ec78",
  "run_id": "8b35bf6c-8ec6-4d31-a970-848063cb7bf4",
  "fault_injection_time": "2026-05-08T05:22:22.000Z",
  "agent_fault_detection_time": "2026-05-08T05:22:43.105Z",
  "agent_fault_mitigation_time": null,
  "time_to_detect": 21.11,
  "time_to_mitigate": null,
  "fault_detected": "No explicit chaos fault was confirmed. The agent observed transient startup probe failures (Unhealthy events for carts-6b5d88f967-qpgk7 on /health with context deadline exceeded and connection refused) and considered the flash-agent pod as a potential external-disturbance/chaos framework but concluded the namespace is healthy or degraded based on pod status and events, without identifying an active injected fault.",
  "detection_success": 1,
  "trajectory_steps": 13,
  "input_tokens": 38603,
  "output_tokens": 12024